# 位置编码 

In [ ]:
import torch
import torch.nn as nn
import math


class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.coa(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor):
        # x.shape: batch_size , seq_len ,d_model
        output = x + self.pe[:, x.size(1)]
        return output

# MultiHeadAttention

In [ ]:
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask=None):
        batch_size = x.size(0)

        Q = self.wq(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.wk(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.wv(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = torch.masked_fill(scores, mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)

        output = (
            torch.matmul(attn_weights, V)
            .transpose(1, 2)
            .contiguous()
            .view(batch_size, -1, self.d_model)
        )

        return self.wo(output)

# GQA

In [ ]:
import torch
import torch.nn as nn
import math


class GQA(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.n_per_group = num_heads // num_kv_heads
        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, num_heads * self.head_dim, bias=False)
        self.Wk = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.Wv = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.Wo = nn.Linear(num_heads * self.head_dim, d_model, bias=False)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        Q = self.Wq(x).view(batch_size, seq_len, -1, self.head_dim).transpose(1, 2)
        K = self.Wk(x).view(batch_size, seq_len, -1, self.head_dim).transpose(1, 2)
        V = self.Wv(x).view(batch_size, seq_len, -1, self.head_dim).transpose(1, 2)

        K = torch.repeat_interleave(K, self.n_per_group, dim=1)
        V = torch.repeat_interleave(V, self.n_per_group, dim=1)

        scores = torch.matmul(Q,K.transpose(-1,-2)) / math.sqrt(self.head_dim)

        attn_weights = torch.softmax(scores,dim=-1)

        # b h s s , b h s d  -> b h s d  ---> b s h d  ---> b s dmodel
        output = torch.matmul(attn_weights,V).transpose(1,2).contiguous().view(batch_size,seq_len,-1)

        return self.Wo(output)


# MoE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


class MoE(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k):
        super().__init__()
        self.d_model = d_model
        self.num_experts = num_experts
        self.top_k = top_k
        self.experts = nn.ModuleList(
            [Expert(d_model, d_ff) for _ in range(num_experts)]
        )
        self.router = nn.Linear(d_model, num_experts)

    def forward(self, x: torch.Tensor):
        original_shape = x.shape
        x_flat = x.view(-1, original_shape[-1])

        logits = self.router(x_flat)
        scores, indices = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(scores, dim=-1)
        output = torch.zeros_like(x_flat)

        for k in range(self.top_k):
            expert_idx_at_k = indices[:, k]
            for i in range(self.num_experts):
                token_mask = expert_idx_at_k == i
                if token_mask.any():
                    output[token_mask] += weights[token_mask, k] * self.experts[i](
                        x_flat[token_mask]
                    )

        return output.view(original_shape)

# RSMNorm

In [1]:
import torch
import torch.nn as nn

class RSMNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.weights= nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self,x:torch.Tensor):
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1,keepdim=True) + self.eps)
        return (x.float() * rms ) *self.weights

/opt/homebrew/Caskroom/miniconda/base/envs/llm/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


# SwiGLU-FFN

In [ ]:
import troch
import torch.nn as nn
import torch.nn.functional as F


class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.v = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        swish_gate = F.silu(self.w1(x))
        linear_path = self.v(x)

        intermediate = swish_gate * linear_path

        return self.w2(intermediate)


# RoPE

In [ ]:
import torch
import torch.nn


def rotate_half(x: torch.Tensor):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        theta_i = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))

        t = torch.arange(max_seq_len).type_as(theta_i)

        freqs = torch.einsum("i,j->ij", t, theta_i)

        emb = torch.cat([freqs, freqs], dim=-1)

        cos_cache = emb.cos()[None, None, :, :]
        sin_cache = emb.sin()[None, None, :, :]

        self.register_buffer("cos_cache", cos_cache)
        self.register_buffer("sin_cache", sin_cache)

    def forward(self, q: torch.Tensor, k: torch.Tensor):
        cos = self.cos_cache[:, :, : q.size(2), :]
        sin = self.sin_cache[:, :, : q.size(2), :]

        q_embed = q * cos + (rotate_half(q) * sin)
        k_embed = k * cos + (rotate_half(k) * sin)

        return q_embed, k_embed